# Ion Channel MD Analysis — GROMACS Post-Processing

Reusable analysis and plotting pipeline for molecular dynamics trajectories of
ion channels, built on the `.xvg` files that GROMACS analysis tools emit.

The notebook computes and plots:

| Observable | GROMACS source | What it tells you |
|---|---|---|
| **RMSD** vs time | `gmx rms` | Global structural drift / equilibration |
| **RMSF** per residue | `gmx rmsf` | Which residues are mobile; averaged over protomers |
| **RMSF variability** | derived | Which residues respond most to a changing condition |
| **Radius of gyration** vs time | `gmx gyrate` | Global compaction / expansion |
| **System energies** vs time | `gmx energy` | Potential, kinetic, total, T, P, density |
| **B-factor PDB** | derived | RMSF variability painted onto a structure for PyMOL/VMD |

The pipeline is written for a **homo-oligomeric channel simulated over a grid of
conditions** (the defaults describe a tetramer across 6 temperatures × 3 salinities),
but every one of those assumptions is a variable in the configuration cell.

---

## Quick start

```bash
pip install -r requirements.txt
jupyter lab GeneralIonChannelAnalysis.ipynb
```

Then either:

1. **Try it immediately** — leave `USE_DEMO_DATA = True` in the configuration cell.
   The notebook generates synthetic `.xvg` files and runs end to end, so you can
   confirm the pipeline works before pointing it at real trajectories.
2. **Use your own data** — set `USE_DEMO_DATA = False` and set `DATA_ROOT`
   (or the `MD_DATA_ROOT` environment variable) to your trajectory directory.

## Expected data layout

The default layout is one directory per simulation condition:

```
<DATA_ROOT>/
├── coarse/
│   ├── 305K05/            # 305 K, 50 mM  (see build_condition_dirname)
│   │   ├── RMSDProtein.xvg
│   │   ├── RMSFProtein.xvg
│   │   ├── RGProtein.xvg
│   │   └── NPTEnergy.xvg
│   ├── 310K05/
│   └── ...
└── atomistic/
    └── ...
```

If your layout differs, you do **not** need to touch the analysis code — override
`build_condition_dirname()` and `build_path()` in the configuration cell.

## Generating the input files

```bash
# Centre the protein and remove jumps first (interactive group selection)
gmx trjconv  -s npt.tpr -f npt.xtc -o centered.xtc -pbc mol -center

# RMSD of protein backbone against the starting structure
gmx rms      -s npt.tpr -f centered.xtc -o RMSDProtein.xvg  -tu ps

# Per-residue RMSF (-res gives one value per residue, which this notebook assumes)
gmx rmsf     -s npt.tpr -f centered.xtc -o RMSFProtein.xvg  -res

# Radius of gyration
gmx gyrate   -s npt.tpr -f centered.xtc -o RGProtein.xvg

# Energies: select LJ(SR), Coulomb(SR), Potential, Kinetic, Total, T, P, Density
#           in that order to match ENERGY_COLUMNS below
gmx energy   -f npt.edr -o NPTEnergy.xvg

# A reference structure for the B-factor output
gmx trjconv  -s npt.tpr -f centered.xtc -o reference.pdb -dump 0
```

## Units

GROMACS writes lengths in **nm** and times in **ps**. This notebook converts on
load, so every array in memory and every axis label is in **ångström** and
**nanoseconds**. Conversion factors live in the configuration cell.

## Adapting to a different channel — checklist

1. `N_PROTOMERS` — 4 for most TRP/K⁺ channels, 5 for pLGICs (nAChR, GLIC), 3 for ASIC/P2X.
2. `RESIDUE_OFFSET` / `RESIDUE_NUMBER_ORIGIN` — align residue numbering with your construct.
3. `TEMPERATURES_K` / `SALINITIES_MM` — or replace with whatever conditions you varied.
4. `DOMAIN_SEGMENTS` — the domain bands drawn under the RMSF plot.
5. `REFERENCE_PDB` — a structure for the B-factor output.
6. `FILENAMES` — if your `.xvg` files are named differently.


---

## 1. Imports

In [ ]:
# ---------------------------------------------------------------------------
# Standard library
# ---------------------------------------------------------------------------
import os
import re
import itertools
from dataclasses import dataclass
from pathlib import Path

# ---------------------------------------------------------------------------
# Third party
# ---------------------------------------------------------------------------
import numpy as np                       # numerical arrays and statistics
import matplotlib.pyplot as plt          # figure generation
from matplotlib.colors import LogNorm    # log colour scaling for variability maps
from matplotlib.patches import Patch     # proxy handles for the domain legend
from matplotlib.ticker import ScalarFormatter  # compact labels for large numbers
from scipy.signal import savgol_filter   # smoothing of noisy time series
from scipy.signal import find_peaks       # peak picking for "critical" residues

# ---------------------------------------------------------------------------
# Shared repository configuration
# ---------------------------------------------------------------------------
# channel_config.py sits beside this notebook and holds everything common to
# every notebook here: the data root, the simulation grid, the channel topology,
# the domain boundaries and the plotting primitives. The star import brings in
# only the shared, read-only names it lists in __all__.
import channel_config as cc
from channel_config import *

# Render figures inline in the notebook
%matplotlib inline

print(f"numpy {np.__version__}")
import scipy, matplotlib
print(f"scipy {scipy.__version__}")
print(f"matplotlib {matplotlib.__version__}")

---

## 2. Configuration

Everything shared with the other notebooks in this repository lives in `channel_config.py`; only this notebook's own settings are here. No path or channel-specific number appears anywhere below this cell.

In [ ]:
# ===========================================================================
#                            C O N F I G U R A T I O N
#
#  Only the settings specific to THIS notebook live here. Everything shared
#  across the repository - the data root, the simulation grid, the channel
#  topology, the domain boundaries and the plot style - lives in
#  channel_config.py, so it can only ever be defined once.
#
#  To change a shared value (protomer count, temperatures, domains, ...), edit
#  channel_config.py and every notebook picks it up.
# ===========================================================================

# ---------------------------------------------------------------------------
# 1. This notebook
# ---------------------------------------------------------------------------
# Set USE_DEMO_DATA = True to generate small synthetic inputs and run the whole
# notebook without real trajectories. Set it to False to use your own data;
# point channel_config.REAL_DATA_ROOT at it, or set the MD_DATA_ROOT
# environment variable.
USE_DEMO_DATA = True

# Names this notebook's output directory, results/<NOTEBOOK_SLUG>/. Change it if
# you copy this notebook as the basis for a new analysis, so the two cannot
# overwrite each other's figures.
NOTEBOOK_SLUG = "general_ion_channel_analysis"

# Write every figure to OUTPUT_DIR as it is drawn.
SAVE_FIGURES = False

# Logical name -> file name inside each condition directory.
FILENAMES = {
    "rmsd": "RMSDProtein.xvg",
    "rmsf": "RMSFProtein.xvg",
    "rg": "RGProtein.xvg",
    "energy": "NPTEnergy.xvg",
}


# Resolved paths.
DATA_ROOT = cc.resolve_data_root(USE_DEMO_DATA)
DEMO_DATA_ROOT = cc.DEMO_DATA_ROOT
OUTPUT_DIR = cc.make_output_dir(NOTEBOOK_SLUG)


def build_path(model_level, temperature_k, salinity_mm, key):
    """Return the full path to one input file.

    Override this if your directory tree is shaped differently, e.g. if the
    model level is a filename suffix rather than a parent directory. The
    condition directory name itself comes from cc.condition_dirname().
    """
    return (
        DATA_ROOT
        / model_level
        / cc.condition_dirname(temperature_k, salinity_mm)
        / FILENAMES[key]
    )


def save_figure(fig, name):
    """Save a figure into this notebook's output directory."""
    return cc.save_figure(fig, name, OUTPUT_DIR, SAVE_FIGURES)


# ---------------------------------------------------------------------------
# 2. Reference structure
# ---------------------------------------------------------------------------
# Structure for the B-factor output. A path such as
# Path("structures/reference.pdb"), or None to skip that section. In demo mode
# this is replaced with a generated structure.
REFERENCE_PDB = None

# ---------------------------------------------------------------------------
# 3. Energy file column layout
# ---------------------------------------------------------------------------
# Column 0 of a gmx energy .xvg is always time. The remaining columns follow the
# order in which terms were selected. Adjust to match your own selection.
ENERGY_COLUMNS = {
    "lj_sr": 1,
    "coulomb_sr": 2,
    "potential": 3,
    "kinetic": 4,
    "total": 5,
    "temperature": 6,
    "pressure": 7,
    "density": 8,
}

# ---------------------------------------------------------------------------
# 4. Smoothing
# ---------------------------------------------------------------------------
# Savitzky-Golay parameters for time series. The window is clamped to the length
# of each trace automatically, so these are safe upper bounds. Smoothing is for
# display only; reported statistics use the raw data.
SMOOTH_WINDOW = 101
SMOOTH_POLYORDER = 3

# ---------------------------------------------------------------------------
# 5. Peak picking for "critical" residues
# ---------------------------------------------------------------------------
# A residue is flagged when its RMSF forms a peak rising at least
# PEAK_PROMINENCE above the surrounding baseline. PEAK_MIN_SEPARATION stops
# adjacent residues in one flexible loop being reported many times over.
PEAK_PROMINENCE = 0.1        # angstrom
PEAK_MIN_SEPARATION = 15     # residues

# ---------------------------------------------------------------------------
# 6. Figure sizes and colour maps
# ---------------------------------------------------------------------------
FIGSIZE_TIMESERIES = (15, 8)
FIGSIZE_PROFILE = (15, 8)
FIGSIZE_HEATMAP = (8, 15)
VARIABILITY_CMAP = "plasma"


# ---------------------------------------------------------------------------
# Apply the shared plot style and report the resolved configuration
# ---------------------------------------------------------------------------
cc.apply_style()
cc.describe_config(DATA_ROOT, USE_DEMO_DATA, OUTPUT_DIR)

---

## 3. The condition grid

Rather than maintaining lists of file paths by hand, the set of simulations is described once and every later selection is a query against it. This is what makes the analysis cells free of hard-coded slices like `[0:6]` or `[12:18]`.

In [ ]:
# The set of simulations, built once from the shared grid definition. Every
# later selection is a query against it - grid.where(salinity_mm=50) returns the
# positions of those runs - so slicing is never hard coded as [0:6] or [12:18].
grid = cc.ConditionGrid(TEMPERATURES_K, SALINITIES_MM, build_path)
grid.describe()

---

## 4. Demo data (optional)

Generates synthetic `.xvg` files so the notebook can be run end to end without real trajectories, which is useful for checking an installation or trying a modification. Set `USE_DEMO_DATA = False` to skip this and use your own data; the cell then reports any expected files that are missing.

In [ ]:
# ===========================================================================
#  Optional: synthetic demo data
#
#  Writes .xvg files with the same structure GROMACS produces, so the whole
#  notebook can be run and tested without access to real trajectories. The
#  numbers are physically plausible in shape and magnitude but carry no
#  scientific meaning whatsoever.
#
#  Skipped entirely when USE_DEMO_DATA is False.
# ===========================================================================

# Trajectory lengths per model level (picoseconds) and number of saved frames.
DEMO_DURATION_PS = {"coarse": 4.0e6, "atomistic": 4.0e5}
DEMO_N_FRAMES = 400
DEMO_RESIDUES_PER_PROTOMER = 556
DEMO_DEFAULT_DURATION_PS = 1.0e6


def _write_xvg(path, columns, title, xaxis, yaxis, legends=()):
    """Write a GROMACS-style .xvg file with @/# header lines."""
    path.parent.mkdir(parents=True, exist_ok=True)
    data = np.column_stack(columns)
    with open(path, "w") as handle:
        handle.write("# Synthetic data generated for demonstration purposes.\n")
        handle.write("# Not derived from any real simulation.\n")
        handle.write(f'@    title "{title}"\n')
        handle.write(f'@    xaxis  label "{xaxis}"\n')
        handle.write(f'@    yaxis  label "{yaxis}"\n')
        for i, legend in enumerate(legends):
            handle.write(f'@ s{i} legend "{legend}"\n')
        for row in data:
            handle.write(" ".join(f"{value:14.6f}" for value in row) + "\n")


def generate_demo_dataset(rng_seed=0):
    """Populate DEMO_DATA_ROOT with a full grid of synthetic .xvg files."""
    rng = np.random.default_rng(rng_seed)
    n_written = 0

    # The same loops are flexible in every run of a real system, so the peak
    # positions are drawn once and reused across all conditions. Their heights
    # then vary with temperature, which is what the variability analysis picks up.
    n_res = DEMO_RESIDUES_PER_PROTOMER
    flexible_loops = np.sort(
        rng.choice(np.arange(20, n_res - 20), size=12, replace=False)
    )

    for model_level in MODEL_LEVELS:
        duration_ps = DEMO_DURATION_PS.get(model_level, DEMO_DEFAULT_DURATION_PS)
        time_ps = np.linspace(0.0, duration_ps, DEMO_N_FRAMES)

        # Residue numbers as this model level would write them, chosen so that
        # adding RESIDUE_OFFSET puts every level on the common numbering scheme.
        first_residue = RESIDUE_NUMBER_ORIGIN - RESIDUE_OFFSET.get(model_level, 0)

        for cond in grid:
            # Warmer and saltier runs drift a little more, purely so the demo
            # plots show a visible trend.
            heat = (cond.temperature_k - min(TEMPERATURES_K)) / 25.0
            salt = cond.salinity_mm / 150.0

            # --- RMSD (nm): exponential rise to a plateau plus noise ---------
            plateau_nm = 0.35 + 0.45 * heat + 0.1 * salt
            rmsd_nm = plateau_nm * (1.0 - np.exp(-time_ps / (0.15 * duration_ps)))
            rmsd_nm += rng.normal(0.0, 0.02, time_ps.size)
            rmsd_nm = np.clip(rmsd_nm, 0.0, None)
            _write_xvg(
                build_path(model_level, cond.temperature_k, cond.salinity_mm, "rmsd"),
                [time_ps, rmsd_nm], "RMSD", "Time (ps)", "RMSD (nm)",
            )

            # --- RMSF (nm): baseline plus loop bumps, one block per protomer --
            # gmx rmsf -res writes all protomers consecutively, so the residue
            # axis runs over the whole assembly.
            residue_axis = np.arange(
                first_residue, first_residue + n_res * N_PROTOMERS
            )
            x = np.arange(n_res)
            per_protomer = []
            for _ in range(N_PROTOMERS):
                profile = 0.08 + 0.05 * heat + rng.normal(0.0, 0.01, n_res)
                for centre in flexible_loops:
                    height = (0.05 + 0.25 * rng.random()) * (1.0 + heat)
                    width = 4.0 + 10.0 * rng.random()
                    profile += height * np.exp(-0.5 * ((x - centre) / width) ** 2)
                per_protomer.append(np.clip(profile, 0.01, None))
            rmsf_nm = np.concatenate(per_protomer)
            _write_xvg(
                build_path(model_level, cond.temperature_k, cond.salinity_mm, "rmsf"),
                [residue_axis, rmsf_nm], "RMSF", "Residue", "RMSF (nm)",
            )

            # --- Radius of gyration (nm) -------------------------------------
            rg_nm = 4.35 + 0.12 * heat + 0.03 * np.sin(time_ps / (0.05 * duration_ps))
            rg_nm += rng.normal(0.0, 0.01, time_ps.size)
            _write_xvg(
                build_path(model_level, cond.temperature_k, cond.salinity_mm, "rg"),
                [time_ps, rg_nm], "Radius of gyration", "Time (ps)", "Rg (nm)",
            )

            # --- Energies: one column per entry in ENERGY_COLUMNS ------------
            scale = 6.0e6 if model_level == "atomistic" else 1.15e6
            potential = -scale * (1.0 - 0.02 * heat) + rng.normal(0.0, scale * 2e-3, time_ps.size)
            kinetic = 0.25 * scale * (1.0 + 0.05 * heat) + rng.normal(0.0, scale * 1e-3, time_ps.size)
            columns_by_name = {
                "lj_sr": 0.3 * potential,
                "coulomb_sr": 0.6 * potential,
                "potential": potential,
                "kinetic": kinetic,
                "total": potential + kinetic,
                "temperature": cond.temperature_k + rng.normal(0.0, 1.0, time_ps.size),
                "pressure": rng.normal(1.0, 25.0, time_ps.size),
                "density": 1010.0 + 0.6 * cond.salinity_mm + rng.normal(0.0, 2.0, time_ps.size),
            }
            n_columns = max(ENERGY_COLUMNS.values()) + 1
            energy_columns = [time_ps] + [np.zeros(time_ps.size)] * (n_columns - 1)
            for name, index in ENERGY_COLUMNS.items():
                energy_columns[index] = columns_by_name.get(name, np.zeros(time_ps.size))
            _write_xvg(
                build_path(model_level, cond.temperature_k, cond.salinity_mm, "energy"),
                energy_columns, "Energies", "Time (ps)", "Energy (kJ/mol)",
                legends=list(ENERGY_COLUMNS),
            )
            n_written += 4

    return n_written


def generate_demo_reference_pdb(path, n_residues, chains=None):
    """Write a minimal CA-only PDB so the B-factor section has an input.

    One CA atom per residue per chain, laid out on a helix. Geometrically
    meaningless, but structurally valid enough for the B-factor writer.
    """
    chains = chains or [chr(ord("A") + i) for i in range(N_PROTOMERS)]
    path.parent.mkdir(parents=True, exist_ok=True)
    serial = 1
    with open(path, "w") as handle:
        handle.write("REMARK   Synthetic CA-only structure for demonstration only.\n")
        for chain_index, chain in enumerate(chains):
            for res in range(1, n_residues + 1):
                angle = 0.1 * res + chain_index * (2.0 * np.pi / len(chains))
                x = 25.0 * np.cos(angle)
                y = 25.0 * np.sin(angle)
                z = 1.5 * res - 0.75 * n_residues
                handle.write(
                    f"ATOM  {serial:5d}  CA  ALA {chain}{res:4d}    "
                    f"{x:8.3f}{y:8.3f}{z:8.3f}  1.00  0.00           C\n"
                )
                serial += 1
        handle.write("END\n")


if USE_DEMO_DATA:
    n_files = generate_demo_dataset()
    demo_pdb = DEMO_DATA_ROOT / "reference.pdb"
    # Cover residues 1 .. (origin + protomer length) so the whole modelled range
    # is present in the structure.
    generate_demo_reference_pdb(
        demo_pdb,
        n_residues=RESIDUE_NUMBER_ORIGIN + DEMO_RESIDUES_PER_PROTOMER - 1,
    )
    REFERENCE_PDB = demo_pdb
    print(f"Wrote {n_files} synthetic .xvg files under {DEMO_DATA_ROOT.resolve()}")
    print(f"Wrote synthetic reference structure {demo_pdb}")
    print("\nThese files are for testing the pipeline only - they are not simulation data.")
else:
    print("Demo data generation skipped (USE_DEMO_DATA is False).")
    for model_level in MODEL_LEVELS:
        for observable in FILENAMES:
            missing = grid.report_missing(model_level, observable)
            if missing:
                print(f"  WARNING {model_level}/{observable}: {len(missing)} file(s) missing, "
                      f"first is {missing[0]}")

---

## 5. Reading `.xvg` files

All four observables share the GROMACS `.xvg` format, so validation, loading, unit conversion and smoothing are implemented once here. The classes in the following sections add only what is genuinely specific to each observable.

In [ ]:
# ===========================================================================
#  Core .xvg reader
#
#  All four observables come from the same file format, so file validation and
#  loading live in one place. The observable-specific classes below subclass
#  this and add only what is genuinely different.
# ===========================================================================

class XVGSeries:
    """Loads a set of GROMACS .xvg files as (x, y) series.

    Parameters
    ----------
    files : str, Path, or sequence of them
        The .xvg files to read. A single path is accepted and wrapped in a list.
    x_column, y_column : int
        Zero-based column indices. Column 0 of a GROMACS .xvg is normally time
        or residue number.
    x_scale, y_scale : float
        Multiplied into the data on load, used here to convert ps to ns and
        nm to angstrom so that all downstream arrays share one unit system.
    labels : sequence of str, optional
        One label per file, used as the default plot legend.

    Attributes (after `load()`)
    ---------------------------
    x, y : list of np.ndarray
        One array per input file, in the order the files were given.
    """

    SUPPORTED_SUFFIXES = {".xvg", ".dat", ".txt"}
    COMMENT_MARKERS = ["@", "#", "&"]

    def __init__(self, files, x_column=0, y_column=1, x_scale=1.0, y_scale=1.0,
                 labels=None):
        self.files = self._validate(files)
        self.x_column = x_column
        self.y_column = y_column
        self.x_scale = x_scale
        self.y_scale = y_scale
        self.labels = list(labels) if labels is not None else [p.name for p in self.files]
        self.x = None
        self.y = None

    # -- validation ---------------------------------------------------------
    @classmethod
    def _validate(cls, files):
        """Normalise the input to a list of Paths and check every file."""
        # Accept a bare string/Path as a one-element list. The original code
        # accepted a string but then iterated over its characters, so a single
        # file could never actually be loaded; wrapping fixes that.
        if isinstance(files, (str, Path)):
            files = [files]
        try:
            paths = [Path(f) for f in files]
        except TypeError as exc:
            raise TypeError(
                "files must be a path or a sequence of paths, "
                f"got {type(files).__name__}"
            ) from exc
        if not paths:
            raise ValueError("No input files given.")

        bad_suffix = [p for p in paths if p.suffix.lower() not in cls.SUPPORTED_SUFFIXES]
        if bad_suffix:
            raise ValueError(
                f"Unsupported file type '{bad_suffix[0].suffix}'. "
                f"Supported: {sorted(cls.SUPPORTED_SUFFIXES)}"
            )

        missing = [p for p in paths if not p.is_file()]
        if missing:
            raise FileNotFoundError(
                f"{len(missing)} of {len(paths)} input file(s) not found. "
                f"First missing: {missing[0]}\n"
                "Check DATA_ROOT and build_path() in the configuration cell."
            )
        return paths

    # -- loading ------------------------------------------------------------
    def load(self):
        """Read every file into `self.x` and `self.y`. Returns self."""
        self.x, self.y = [], []
        for path in self.files:
            data = np.loadtxt(path, comments=self.COMMENT_MARKERS, ndmin=2)
            n_columns = data.shape[1]
            if max(self.x_column, self.y_column) >= n_columns:
                raise ValueError(
                    f"{path.name} has {n_columns} column(s); cannot read column "
                    f"{max(self.x_column, self.y_column)}. "
                    "Check the column indices (ENERGY_COLUMNS for energy files)."
                )
            self.x.append(data[:, self.x_column] * self.x_scale)
            self.y.append(data[:, self.y_column] * self.y_scale)
        return self

    def _ensure_loaded(self):
        if self.y is None:
            self.load()

    # -- processing ---------------------------------------------------------
    def smooth(self, window_length=None, polyorder=None):
        """Savitzky-Golay smoothing; result in `self.y_smooth`. Returns self.

        The window is clamped to each trace individually, so short traces are
        handled without error. savgol_filter requires an odd window that is
        longer than the polynomial order.
        """
        window_length = SMOOTH_WINDOW if window_length is None else window_length
        polyorder = SMOOTH_POLYORDER if polyorder is None else polyorder
        self._ensure_loaded()

        self.y_smooth = []
        for trace in self.y:
            window = max(min(window_length, len(trace) - 1), polyorder + 2)
            if window % 2 == 0:
                window -= 1
            if window <= polyorder:
                # Too short to smooth meaningfully; keep the raw trace.
                self.y_smooth.append(np.asarray(trace))
                continue
            self.y_smooth.append(savgol_filter(trace, window_length=window,
                                               polyorder=polyorder))
        return self

    # -- reporting ----------------------------------------------------------
    def summary(self, value_name="value", indices=None, labels=None):
        """Print mean, standard deviation and range for each loaded series."""
        self._ensure_loaded()
        indices = range(len(self.y)) if indices is None else indices
        labels = self.labels if labels is None else labels
        header = f"{'series':<22}{'mean':>12}{'std':>12}{'min':>12}{'max':>12}"
        print(f"{value_name}\n{header}\n{'-' * len(header)}")
        for i in indices:
            trace = np.asarray(self.y[i])
            label = str(labels[i]) if i < len(labels) else f"[{i}]"
            print(f"{label:<22}{trace.mean():12.3f}{trace.std():12.3f}"
                  f"{trace.min():12.3f}{trace.max():12.3f}")

---

## 6. RMSD and radius of gyration

In [ ]:
# ===========================================================================
#  RMSD and radius of gyration
#
#  Both are a single scalar per frame, so both are thin wrappers over XVGSeries
#  that fix the unit conversion (ps -> ns, nm -> A) and expose readable
#  attribute names.
# ===========================================================================

class RMSDAnalysis(XVGSeries):
    """Root-mean-square deviation of the protein against a reference frame.

    Produced by ``gmx rms``. Input is in nm and ps; the loaded arrays are in
    angstrom and nanoseconds.

    Example
    -------
    >>> rmsd = RMSDAnalysis(grid.paths("atomistic", "rmsd"),
    ...                     labels=grid.labels(by="both")).smooth()
    >>> rmsd.time[0].max(), rmsd.smoothed_rmsd[0].mean()
    """

    def __init__(self, files, labels=None):
        super().__init__(files, x_column=0, y_column=1,
                         x_scale=PS_TO_NS, y_scale=NM_TO_ANGSTROM, labels=labels)

    @property
    def time(self):
        """Simulation time in ns, one array per file."""
        self._ensure_loaded()
        return self.x

    @property
    def rmsd(self):
        """Raw RMSD in angstrom, one array per file."""
        self._ensure_loaded()
        return self.y

    @property
    def smoothed_rmsd(self):
        """Savitzky-Golay smoothed RMSD in angstrom."""
        if not hasattr(self, "y_smooth"):
            self.smooth()
        return self.y_smooth


class RGAnalysis(XVGSeries):
    """Radius of gyration of the protein.

    Produced by ``gmx gyrate``, whose column 1 is the total Rg (columns 2-4 are
    the x, y and z components, available via `y_column` if wanted).
    """

    def __init__(self, files, labels=None, component=None):
        # component: None -> total Rg; "x", "y", "z" -> columns 2, 3, 4
        column = {None: 1, "x": 2, "y": 3, "z": 4}[component]
        super().__init__(files, x_column=0, y_column=column,
                         x_scale=PS_TO_NS, y_scale=NM_TO_ANGSTROM, labels=labels)

    @property
    def time(self):
        self._ensure_loaded()
        return self.x

    @property
    def rg(self):
        self._ensure_loaded()
        return self.y

    @property
    def smoothed_rg(self):
        if not hasattr(self, "y_smooth"):
            self.smooth()
        return self.y_smooth

---

## 7. RMSF

`gmx rmsf -res` reports one value per residue for the entire assembly. For a homo-oligomer the useful quantity is the average across subunits, giving a single profile for one protomer.

In [ ]:
# ===========================================================================
#  RMSF
#
#  gmx rmsf -res writes one value per residue for the whole assembly, so a
#  tetramer gives 4 x N values. For a homo-oligomer the informative quantity is
#  the average over protomers, which is what `average_over_protomers` produces.
# ===========================================================================

class RMSFAnalysis(XVGSeries):
    """Per-residue root-mean-square fluctuation.

    Produced by ``gmx rmsf -res``. Values are converted from nm to angstrom on
    load; the x column is a residue number and is not scaled.
    """

    def __init__(self, files, labels=None):
        super().__init__(files, x_column=0, y_column=1,
                         x_scale=1.0, y_scale=NM_TO_ANGSTROM, labels=labels)

    @property
    def residues_all(self):
        """Residue numbers as written by GROMACS (all protomers)."""
        self._ensure_loaded()
        return self.x

    @property
    def rmsf_all(self):
        """RMSF in angstrom for all protomers, unaveraged."""
        self._ensure_loaded()
        return self.y

    # -- protomer averaging -------------------------------------------------
    def average_over_protomers(self, n_protomers=None):
        """Average the RMSF profile across identical subunits.

        Assumes the residues of each protomer appear as consecutive equal-length
        blocks, which is how ``gmx rmsf`` orders a multi-chain selection.

        Sets `self.profiles` (one averaged profile per file) and
        `self.protomer_residues` (the residue numbers of the first subunit, as
        written in the file). Returns self.
        """
        n_protomers = N_PROTOMERS if n_protomers is None else n_protomers
        self._ensure_loaded()

        self.profiles = []
        self.protomer_residues = []
        for path, residues, values in zip(self.files, self.x, self.y):
            if len(values) % n_protomers != 0:
                raise ValueError(
                    f"{path.name}: {len(values)} residues is not divisible by "
                    f"n_protomers={n_protomers}. Either the selection used for "
                    "gmx rmsf did not cover whole subunits, or N_PROTOMERS is wrong."
                )
            n_res = len(values) // n_protomers
            # Reshape to (n_protomers, n_res) and collapse the protomer axis.
            # This replaces the original nested Python loop and is equivalent.
            profile = values.reshape(n_protomers, n_res).mean(axis=0)
            self.profiles.append(profile)
            # Keep the first subunit's numbering as the profile's residue axis.
            self.protomer_residues.append(residues[:n_res].astype(int))
        return self

    def residue_numbers(self, index=0, residue_offset=0):
        """Residue numbers for one averaged profile, on the common scheme.

        `residue_offset` should be RESIDUE_OFFSET[model_level], which shifts the
        numbering written by that model onto the reference numbering.
        """
        self._ensure_profiles()
        return self.protomer_residues[index] + residue_offset

    def _ensure_profiles(self):
        if not hasattr(self, "profiles"):
            self.average_over_protomers()

    # -- peak picking -------------------------------------------------------
    def critical_residues(self, prominence=None, min_separation=None):
        """Identify residues sitting at prominent RMSF peaks.

        These are candidate hinge or gating-loop residues: positions that
        fluctuate markedly more than their local neighbourhood.

        `prominence` is how far (in angstrom) a peak must rise above the
        surrounding baseline; `min_separation` stops one broad flexible loop
        being reported as many adjacent hits.

        Sets `self.peaks` (0-based indices into the averaged profile) and
        `self.peak_properties`. Returns self.
        """
        prominence = PEAK_PROMINENCE if prominence is None else prominence
        min_separation = PEAK_MIN_SEPARATION if min_separation is None else min_separation
        self._ensure_profiles()

        self.peaks = []
        self.peak_properties = []
        for profile in self.profiles:
            peaks, properties = find_peaks(
                np.asarray(profile), prominence=prominence, distance=min_separation
            )
            self.peaks.append(peaks)
            self.peak_properties.append(properties)
        return self

    def peak_residue_numbers(self, index, residue_offset=0):
        """Return the peak positions for one file as real residue numbers."""
        if not hasattr(self, "peaks"):
            self.critical_residues()
        return self.residue_numbers(index, residue_offset)[self.peaks[index]]

    # -- variability across a set of conditions -----------------------------
    def variability(self, indices):
        """Standard deviation of the RMSF profile across selected conditions.

        A high value at a residue means that residue's mobility is sensitive to
        whichever variable distinguishes the selected simulations.
        """
        self._ensure_profiles()
        selected = [np.asarray(self.profiles[i]) for i in indices]
        lengths = {len(p) for p in selected}
        if len(lengths) != 1:
            raise ValueError(
                f"Profiles have differing lengths {sorted(lengths)}; they must "
                "cover the same residues to be compared."
            )
        return np.std(np.vstack(selected), axis=0)

---

## 8. System energies

In [ ]:
# ===========================================================================
#  System energies
# ===========================================================================

class EnergyAnalysis(XVGSeries):
    """Thermodynamic quantities written by ``gmx energy``.

    Which physical quantity sits in which column depends on the order terms were
    selected when the file was made, so the mapping is configurable via
    ENERGY_COLUMNS rather than assumed.

    Example
    -------
    >>> energy = EnergyAnalysis(grid.paths("atomistic", "energy"), term="potential")
    >>> energy.load().values[0].mean()
    """

    def __init__(self, files, term="potential", labels=None, columns=None):
        columns = ENERGY_COLUMNS if columns is None else columns
        if term not in columns:
            raise KeyError(
                f"Unknown energy term '{term}'. Available: {sorted(columns)}. "
                "Add it to ENERGY_COLUMNS if your .xvg contains other terms."
            )
        self.term = term
        super().__init__(files, x_column=0, y_column=columns[term],
                         x_scale=PS_TO_NS, y_scale=1.0, labels=labels)

    @property
    def time(self):
        self._ensure_loaded()
        return self.x

    @property
    def values(self):
        """The selected energy term, one array per file (kJ/mol, K, bar or kg/m3)."""
        self._ensure_loaded()
        return self.y

    @property
    def axis_label(self):
        """A sensible y-axis label for the selected term."""
        labels = {
            "lj_sr": "LJ (SR) (kJ/mol)",
            "coulomb_sr": "Coulomb (SR) (kJ/mol)",
            "potential": "Potential Energy (kJ/mol)",
            "kinetic": "Kinetic Energy (kJ/mol)",
            "total": "Total Energy (kJ/mol)",
            "temperature": "Temperature (K)",
            "pressure": "Pressure (bar)",
            "density": r"Density (kg m$^{-3}$)",
        }
        return labels.get(self.term, self.term.replace("_", " ").title())

---

## 9. Mapping variability onto a structure

Writes a copy of a reference PDB with the B-factor column replaced by a per-residue score, so that variability can be viewed on the structure in PyMOL or VMD.

In [ ]:
# ===========================================================================
#  Painting RMSF variability onto a structure
#
#  Writes a copy of a reference PDB with the B-factor column replaced by a
#  per-residue score. Open the result in PyMOL and colour by B-factor:
#
#      load NPTRMSFVariability_coarse_50mM.pdb
#      spectrum b, blue_white_red
#
#  so that the residues most sensitive to the varied condition are highlighted.
# ===========================================================================

# PDB fixed-column format: the B-factor field is columns 61-66 (1-based),
# i.e. characters [60:66] of the line.
PDB_BFACTOR_SLICE = slice(60, 66)
PDB_RESSEQ_SLICE = slice(22, 26)


def scale_to_bfactor(values, log_transform=True, floor=1e-6):
    """Rescale arbitrary per-residue values onto the 0-100 B-factor range.

    RMSF variability spans orders of magnitude, so a log transform before
    normalising usually gives a more readable colour spread. Values at or below
    zero are clipped to `floor` first, since log(0) is undefined - the original
    implementation passed raw standard deviations to np.log and could produce
    -inf for a perfectly reproducible residue.
    """
    values = np.asarray(values, dtype=float)
    if log_transform:
        n_clipped = int(np.sum(values <= floor))
        if n_clipped:
            print(f"  note: clipped {n_clipped} non-positive value(s) to {floor:g} "
                  "before the log transform")
        values = np.log(np.clip(values, floor, None))
    span = values.max() - values.min()
    if span == 0:
        # Uniform input: everything maps to the middle of the range.
        return np.full_like(values, 50.0)
    return (values - values.min()) / span * 100.0


def write_bfactor_pdb(values, residue_numbers, reference_pdb, output_pdb,
                      log_transform=True, default_bfactor=0.0):
    """Copy `reference_pdb` to `output_pdb` with B-factors set from `values`.

    Parameters
    ----------
    values : 1-D array
        One score per residue of a single protomer.
    residue_numbers : 1-D array
        The residue number in the reference structure that each entry of
        `values` belongs to. Get this from
        ``RMSFAnalysis.residue_numbers(index, RESIDUE_OFFSET[model_level])``
        so that coarse and atomistic models land on the same numbering.
    reference_pdb : path
        Structure to copy. Its coordinates are untouched.
    output_pdb : path
        Destination.
    log_transform : bool
        Passed to `scale_to_bfactor`.
    default_bfactor : float
        Assigned to residues in the structure that have no matching value
        (e.g. lipids, ions, or residues outside the modelled range).

    Because the lookup is keyed on residue number alone, every protomer in the
    structure receives the same profile - which is the intent, since `values`
    is already a protomer average.

    Returns the number of atoms whose B-factor was assigned a real value.
    """
    reference_pdb = Path(reference_pdb)
    output_pdb = Path(output_pdb)
    if not reference_pdb.is_file():
        raise FileNotFoundError(
            f"Reference structure not found: {reference_pdb}. "
            "Set REFERENCE_PDB in the configuration cell."
        )
    values = np.asarray(values, dtype=float)
    residue_numbers = np.asarray(residue_numbers, dtype=int)
    if values.shape != residue_numbers.shape:
        raise ValueError(
            f"values has {values.size} entries but residue_numbers has "
            f"{residue_numbers.size}; they must correspond one to one."
        )
    output_pdb.parent.mkdir(parents=True, exist_ok=True)

    bfactors = scale_to_bfactor(values, log_transform=log_transform)
    by_residue = {int(res): b for res, b in zip(residue_numbers, bfactors)}

    n_assigned = 0
    n_default = 0
    with open(reference_pdb) as fin, open(output_pdb, "w") as fout:
        for line in fin:
            if not line.startswith(("ATOM", "HETATM")):
                fout.write(line)
                continue
            # Pad short lines so the fixed-column slicing below is always valid.
            line = line.rstrip("\n").ljust(80)
            try:
                res_id = int(line[PDB_RESSEQ_SLICE])
            except ValueError:
                fout.write(line.rstrip() + "\n")
                continue
            if res_id in by_residue:
                bfactor = by_residue[res_id]
                n_assigned += 1
            else:
                bfactor = default_bfactor
                n_default += 1
            fout.write(f"{line[:60]}{bfactor:6.2f}{line[66:]}".rstrip() + "\n")

    print(f"  {output_pdb}: {n_assigned} atom(s) assigned, {n_default} left at "
          f"{default_bfactor:.1f}")
    return n_assigned


def write_variability_pdb_from_files(xvg_files, reference_pdb, output_pdb,
                                     n_protomers=None, residue_offset=0,
                                     log_transform=True):
    """Convenience wrapper: RMSF .xvg files in, B-factor PDB out.

    Loads the given RMSF files, averages each over protomers, takes the
    standard deviation across files, and writes the result as B-factors.
    """
    analysis = RMSFAnalysis(xvg_files).average_over_protomers(n_protomers)
    variability = analysis.variability(range(len(analysis.files)))
    write_bfactor_pdb(
        values=variability,
        residue_numbers=analysis.residue_numbers(0, residue_offset),
        reference_pdb=reference_pdb,
        output_pdb=output_pdb,
        log_transform=log_transform,
    )
    return variability

---

## 10. Plotting

The shared primitives come from `channel_config.py`; these are the observable-specific figures. Tick positions are computed from the data, so they work unchanged whether a trajectory runs for 400 ns or 4 microseconds. Each returns `(fig, ax)` for further customisation.

In [ ]:
# ===========================================================================
#  Observable-specific plotting
#
#  The shared primitives (nice_ticks, apply_ticks, decade_ticks, resolve_colors,
#  add_domain_bands) come from channel_config.py.
# ===========================================================================

def plot_vs_time(times, values, labels=None, ylabel="value", colors=None,
                 legend_title=None, title=None, xlabel="Time (ns)",
                 show_legend=True, ylim=None, ax=None, figure_name=None):
    """Plot one or more traces against time.

    Parameters
    ----------
    times, values : sequences of 1-D arrays
        Matched lists; `times[i]` and `values[i]` must be the same length.
    labels : sequence of str, optional
        Legend entries, one per trace.
    ylim : (low, high), optional
        Fixed y limits. Useful for comparing panels on a common scale.
    """
    mismatched = [
        i for i, (t, v) in enumerate(zip(times, values)) if len(t) != len(v)
    ]
    if mismatched:
        raise ValueError(
            f"Series {mismatched} have differing time and value lengths. "
            "Check that the .xvg files come from the same run."
        )

    fig, ax = (ax.figure, ax) if ax is not None else plt.subplots(figsize=FIGSIZE_TIMESERIES)
    colors = resolve_colors(colors, len(values))
    labels = labels if labels is not None else [None] * len(values)

    for time, value, color, label in zip(times, values, colors, labels):
        ax.plot(time, value, color=color, label=label, linewidth=2)

    x_lo = min(float(np.min(t)) for t in times)
    x_hi = max(float(np.max(t)) for t in times)
    ax.set_xlim(x_lo, x_hi)
    apply_ticks(ax, "x", x_lo, x_hi)

    if ylim is not None:
        ax.set_ylim(*ylim)
    y_lo, y_hi = ax.get_ylim()
    apply_ticks(ax, "y", y_lo, y_hi, n_ticks=7)

    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    if title:
        ax.set_title(title, fontsize=FONTSIZE)
    if show_legend and any(l is not None for l in labels):
        ax.legend(title=legend_title, loc="best", ncol=3,
                  title_fontsize=FONTSIZE * 0.8, fontsize=FONTSIZE * 0.8)
    ax.tick_params(axis="both", which="major", pad=8)

    if figure_name:
        save_figure(fig, figure_name)
    return fig, ax


def plot_rmsf(residues, profiles, labels=None, colors=None,
              legend_title=None, title=None, ylim=None, segments=None,
              show_domain_legend=False, ax=None, figure_name=None):
    """Plot per-residue RMSF profiles with optional domain bands underneath.

    Parameters
    ----------
    residues : 1-D array
        Real residue numbers, from
        ``RMSFAnalysis.residue_numbers(i, RESIDUE_OFFSET[model_level])``.
    profiles : sequence of 1-D arrays
        Protomer-averaged RMSF profiles, all covering `residues`.
    """
    lengths = {len(p) for p in profiles} | {len(residues)}
    if len(lengths) != 1:
        raise ValueError(
            f"RMSF profiles and the residue axis have differing lengths "
            f"{sorted(lengths)}; they must cover the same residues to be overlaid."
        )
    residues = np.asarray(residues)

    fig, ax = (ax.figure, ax) if ax is not None else plt.subplots(figsize=FIGSIZE_PROFILE)
    colors = resolve_colors(colors, len(profiles))
    labels = labels if labels is not None else [None] * len(profiles)

    for profile, color, label in zip(profiles, colors, labels):
        ax.plot(residues, profile, color=color, label=label, linewidth=2)

    ax.set_xlim(residues[0], residues[-1])
    apply_ticks(ax, "x", float(residues[0]), float(residues[-1]), n_ticks=9)

    if ylim is not None:
        ax.set_ylim(*ylim)
    else:
        ax.set_ylim(0, None)
    y_lo, y_hi = ax.get_ylim()
    apply_ticks(ax, "y", y_lo, y_hi, n_ticks=7)

    ax.set_xlabel("Residue Number")
    ax.set_ylabel(r"RMSF ($\mathrm{\AA}$)")
    if title:
        ax.set_title(title, fontsize=FONTSIZE)
    if any(l is not None for l in labels):
        ax.legend(title=legend_title, loc="upper left", ncol=3,
                  title_fontsize=FONTSIZE * 0.8, fontsize=FONTSIZE * 0.8)
    ax.tick_params(axis="both", which="major", pad=10)

    add_domain_bands(ax, segments=segments, show_legend=show_domain_legend)

    if figure_name:
        save_figure(fig, figure_name)
    return fig, ax


def plot_rmsf_variability(residues, columns, column_labels, xlabel,
                          cmap=None, vmin=None, vmax=None, cbar_label=None,
                          title=None, figure_name=None):
    """Heat map of per-residue RMSF variability, one column per group.

    Each column is the standard deviation of the RMSF profile across a set of
    simulations, so bright rows are residues whose mobility depends strongly on
    the variable being scanned.

    A log colour scale is used because the spread typically covers two or three
    orders of magnitude. Drawn with pcolormesh rather than one rectangle per
    residue, which is far faster for long constructs.
    """
    lengths = {len(c) for c in columns} | {len(residues)}
    if len(lengths) != 1:
        raise ValueError(
            f"Variability columns and the residue axis have differing lengths "
            f"{sorted(lengths)}."
        )
    residues = np.asarray(residues)

    matrix = np.vstack([np.asarray(c, dtype=float) for c in columns]).T  # (res, group)
    positive = matrix[matrix > 0]
    if positive.size == 0:
        raise ValueError("All variability values are zero; nothing to plot.")
    vmin = float(positive.min()) if vmin is None else vmin
    vmax = float(matrix.max()) if vmax is None else vmax
    if vmax <= vmin:
        vmax = vmin * 10.0
    matrix = np.clip(matrix, vmin, vmax)

    fig, ax = plt.subplots(figsize=FIGSIZE_HEATMAP)
    x_edges = np.arange(len(columns) + 1) - 0.5
    y_edges = np.append(residues, residues[-1] + 1) - 0.5
    mesh = ax.pcolormesh(
        x_edges, y_edges, matrix,
        cmap=cmap or VARIABILITY_CMAP,
        norm=LogNorm(vmin=vmin, vmax=vmax),
        shading="flat",
    )

    ticks, tick_labels = decade_ticks(vmin, vmax)
    cbar = fig.colorbar(mesh, ax=ax, ticks=ticks, pad=0.02)
    cbar.set_label(cbar_label or r"RMSF SD ($\mathrm{\AA}$)")
    cbar.ax.set_yticklabels(tick_labels)
    cbar.ax.tick_params(labelsize=FONTSIZE)
    cbar.minorticks_off()

    ax.set_xlabel(xlabel)
    ax.set_ylabel("Residue Number")
    ax.set_xticks(range(len(columns)))
    ax.set_xticklabels(column_labels)
    apply_ticks(ax, "y", float(y_edges[0]), float(y_edges[-1]), n_ticks=9)
    if title:
        ax.set_title(title, fontsize=FONTSIZE)

    if figure_name:
        save_figure(fig, figure_name)
    return fig, ax

---

# Analysis

Everything above defines the pipeline; everything below runs it.

---

## 11. Load the data

In [ ]:
# ===========================================================================
#  Load every observable for every model level
#
#  `data[model_level][observable]` holds one analysis object, whose per-run lists
#  follow the order of `grid`. Because the file lists come from the grid, the
#  index of a given run is the same in all four observables.
# ===========================================================================

CONDITION_LABELS = grid.labels(by="both")

data = {}
for model_level in MODEL_LEVELS:
    print(f"Loading {model_level} ...")

    rmsd = RMSDAnalysis(grid.paths(model_level, "rmsd"),
                        labels=CONDITION_LABELS).smooth()

    rmsf = (RMSFAnalysis(grid.paths(model_level, "rmsf"), labels=CONDITION_LABELS)
            .average_over_protomers()
            .critical_residues())

    rg = RGAnalysis(grid.paths(model_level, "rg"), labels=CONDITION_LABELS).smooth()

    energy = EnergyAnalysis(grid.paths(model_level, "energy"),
                            term="potential", labels=CONDITION_LABELS).load()

    data[model_level] = {"rmsd": rmsd, "rmsf": rmsf, "rg": rg, "energy": energy}

    n_frames = len(rmsd.time[0])
    n_res = len(rmsf.profiles[0])
    print(f"  {len(rmsd.files)} runs | {n_frames} frames | "
          f"{rmsd.time[0].max():.0f} ns | {n_res} residues per protomer")

print("\nDone.")

In [ ]:
# Numerical summary of the loaded data, in angstrom / kJ per mol.
# A quick sanity check before plotting: look for runs whose mean RMSD is far
# from the rest, which usually means an unequilibrated or crashed trajectory.
for model_level in MODEL_LEVELS:
    print(f"\n{'=' * 70}\n{model_level.upper()}\n{'=' * 70}")
    data[model_level]["rmsd"].summary(value_name="RMSD (A)")
    print()
    data[model_level]["rg"].summary(value_name="Radius of gyration (A)")

---

## 12. RMSD

Structural drift relative to the starting structure. A trace that plateaus indicates the run has equilibrated; one still climbing at the end has not.

In [ ]:
# One panel per (model level, salinity), with all temperatures overlaid.
# The condition grid supplies the indices, so this loop adapts automatically if
# TEMPERATURES_K or SALINITIES_MM change.
for model_level in MODEL_LEVELS:
    rmsd = data[model_level]["rmsd"]
    for salinity in SALINITIES_MM:
        indices = grid.where(salinity_mm=salinity)
        plot_vs_time(
            times=[rmsd.time[i] for i in indices],
            values=[rmsd.smoothed_rmsd[i] for i in indices],
            labels=grid.labels(indices, by="temperature"),
            ylabel=r"RMSD ($\mathrm{\AA}$)",
            legend_title="Temperature",
            title=f"{model_level.title()} - {salinity:.0f} mM",
            figure_name=f"rmsd_{model_level}_{salinity:.0f}mM",
        )
        plt.show()

---

## 13. RMSF

Per-residue mobility, averaged over protomers. Peaks mark flexible loops and termini; troughs mark the rigid core, typically the transmembrane helices.

In [ ]:
# Protomer-averaged RMSF profiles, one panel per (model level, salinity).
# The coloured strip along the bottom marks the structural domains configured in
# DOMAIN_SEGMENTS. Set show_domain_legend=True to label them.
for model_level in MODEL_LEVELS:
    rmsf = data[model_level]["rmsf"]
    # Residue numbers come from the .xvg files, shifted onto the common
    # numbering scheme by this model level's offset.
    residues = rmsf.residue_numbers(0, RESIDUE_OFFSET.get(model_level, 0))
    for salinity in SALINITIES_MM:
        indices = grid.where(salinity_mm=salinity)
        plot_rmsf(
            residues=residues,
            profiles=[rmsf.profiles[i] for i in indices],
            labels=grid.labels(indices, by="temperature"),
            legend_title="Temperature",
            title=f"{model_level.title()} - {salinity:.0f} mM",
            show_domain_legend=False,
            figure_name=f"rmsf_{model_level}_{salinity:.0f}mM",
        )
        plt.show()

---

## 14. RMSF variability

Standard deviation of the RMSF profile across a set of runs, which isolates the residues whose flexibility actually responds to the variable being scanned rather than those that are simply mobile in every run.

In [ ]:
# How sensitive is each residue's mobility to a change in conditions?
#
#   "across temperature" : SD over all temperatures, one column per salinity
#   "across salinity"    : SD over all salinities, one column per temperature
#
# A bright horizontal band means that residue's flexibility responds strongly to
# the variable being scanned - a candidate allosteric or gating element.
variability = {}

for model_level in MODEL_LEVELS:
    rmsf = data[model_level]["rmsf"]
    residues = rmsf.residue_numbers(0, RESIDUE_OFFSET.get(model_level, 0))

    across_temperature = [
        rmsf.variability(grid.where(salinity_mm=s)) for s in SALINITIES_MM
    ]
    across_salinity = [
        rmsf.variability(grid.where(temperature_k=t)) for t in TEMPERATURES_K
    ]
    variability[model_level] = {
        "residues": residues,
        "across_temperature": across_temperature,
        "across_salinity": across_salinity,
    }

    plot_rmsf_variability(
        residues=residues,
        columns=across_temperature,
        column_labels=[f"{s:.0f}" for s in SALINITIES_MM],
        xlabel="Salinity (mM)",
        title=f"{model_level.title()}: SD across temperature",
        figure_name=f"rmsf_variability_across_temperature_{model_level}",
    )
    plt.show()

    plot_rmsf_variability(
        residues=residues,
        columns=across_salinity,
        column_labels=[f"{t:.0f}" for t in TEMPERATURES_K],
        xlabel="Temperature (K)",
        title=f"{model_level.title()}: SD across salinity",
        figure_name=f"rmsf_variability_across_salinity_{model_level}",
    )
    plt.show()

---

## 15. Candidate critical residues

Prominent RMSF peaks, reported as residue numbers and annotated with the domain they fall in. Positions recurring across most runs are the ones worth following up.

In [ ]:
# Residues sitting at prominent RMSF peaks, reported as real residue numbers.
#
# Tighten PEAK_PROMINENCE to report fewer, sharper peaks; increase
# PEAK_MIN_SEPARATION to avoid reporting several residues from one flexible loop.
for model_level in MODEL_LEVELS:
    rmsf = data[model_level]["rmsf"]
    offset = RESIDUE_OFFSET.get(model_level, 0)
    n_runs = len(rmsf.peaks)
    print(f"\n{'=' * 78}\n{model_level.upper()}  "
          f"(prominence >= {PEAK_PROMINENCE} A, separation >= {PEAK_MIN_SEPARATION} res)"
          f"\n{'=' * 78}")

    # Count how many runs put a peak at each residue. Positions that recur are
    # more trustworthy than one-offs, which are usually noise.
    counts = {}
    for run_index in range(n_runs):
        for residue in rmsf.peak_residue_numbers(run_index, offset):
            counts[int(residue)] = counts.get(int(residue), 0) + 1

    consensus = sorted(res for res, c in counts.items() if c >= max(2, n_runs // 2))

    print(f"{'residue':>9}{'runs':>8}   domain")
    print("-" * 78)
    for residue in consensus:
        print(f"{residue:>9}{counts[residue]:>5}/{n_runs:<2}   "
              f"{describe_domain(residue)}")
    print(f"\n{len(consensus)} consensus peak(s) from "
          f"{len(counts)} distinct position(s) seen at least once.")

---

## 16. Variability on the structure

In [ ]:
# Write RMSF-variability B-factor structures, one per (model level, salinity).
#
# Requires REFERENCE_PDB. In PyMOL:
#     load results/rmsf_variability_coarse_50mM.pdb
#     spectrum b, blue_white_red
if REFERENCE_PDB is None:
    print("REFERENCE_PDB is not set - skipping. Set it in the configuration cell "
          "to a structure whose residue numbering matches your simulations.")
else:
    for model_level in MODEL_LEVELS:
        offset = RESIDUE_OFFSET.get(model_level, 0)
        residues = variability[model_level]["residues"]
        print(f"{model_level} (residue offset {offset}, "
              f"residues {residues.min()}-{residues.max()}):")
        for salinity in SALINITIES_MM:
            values = variability[model_level]["across_temperature"][
                SALINITIES_MM.index(salinity)
            ]
            write_bfactor_pdb(
                values=values,
                residue_numbers=residues,
                reference_pdb=REFERENCE_PDB,
                output_pdb=OUTPUT_DIR / f"rmsf_variability_{model_level}_{salinity:.0f}mM.pdb",
                log_transform=True,
            )

---

## 17. Radius of gyration

In [ ]:
# Radius of gyration: a global measure of expansion or compaction. A steady
# upward drift alongside rising RMSD usually indicates the assembly is loosening
# rather than simply relaxing.
for model_level in MODEL_LEVELS:
    rg = data[model_level]["rg"]
    for salinity in SALINITIES_MM:
        indices = grid.where(salinity_mm=salinity)
        plot_vs_time(
            times=[rg.time[i] for i in indices],
            values=[rg.smoothed_rg[i] for i in indices],
            labels=grid.labels(indices, by="temperature"),
            ylabel=r"$R_\mathrm{g}$ ($\mathrm{\AA}$)",
            legend_title="Temperature",
            title=f"{model_level.title()} - {salinity:.0f} mM",
            figure_name=f"rg_{model_level}_{salinity:.0f}mM",
        )
        plt.show()

---

## 18. System energies

In [ ]:
# Potential energy against time. Mostly an equilibration diagnostic: after the
# initial relaxation the trace should be flat, with a temperature-dependent
# offset between runs.
#
# Change ENERGY_TERM to any key of ENERGY_COLUMNS ("total", "temperature",
# "pressure", "density", ...) to check a different quantity.
ENERGY_TERM = "potential"

for model_level in MODEL_LEVELS:
    energy = EnergyAnalysis(grid.paths(model_level, "energy"),
                            term=ENERGY_TERM, labels=CONDITION_LABELS).load()
    for salinity in SALINITIES_MM:
        indices = grid.where(salinity_mm=salinity)
        plot_vs_time(
            times=[energy.time[i] for i in indices],
            values=[energy.values[i] for i in indices],
            labels=grid.labels(indices, by="temperature"),
            ylabel=energy.axis_label,
            legend_title="Temperature",
            title=f"{model_level.title()} - {salinity:.0f} mM",
            figure_name=f"{ENERGY_TERM}_{model_level}_{salinity:.0f}mM",
        )
        plt.show()

---

## Troubleshooting

**`FileNotFoundError` when loading**
The paths are built by `build_path()` in the configuration cell. Print a sample
to see what the notebook is looking for:

```python
print(grid.paths("atomistic", "rmsd")[0])
```

Then either fix `DATA_ROOT` / `MD_DATA_ROOT`, or override `build_condition_dirname()`
and `build_path()` to match your directory layout.

**`ValueError: N residues is not divisible by n_protomers`**
Either `N_PROTOMERS` does not match your assembly, or the index group used for
`gmx rmsf` did not cover whole subunits. Check the residue count:

```python
rmsf = RMSFAnalysis(grid.paths("atomistic", "rmsf")).load()
print(len(rmsf.rmsf_all[0]), "residues total")
```

**RMSF profiles have differing lengths**
The `.xvg` files were made from different index groups or different constructs.
All runs being compared must cover the same residues.

**Domain bands sit in the wrong place, or the residue axis starts at 1**
The residue axis is read from the `.xvg` files and then shifted by
`RESIDUE_OFFSET[model_level]`. Check what a model level actually writes:

```python
rmsf = RMSFAnalysis(grid.paths("coarse", "rmsf")).average_over_protomers()
print(rmsf.protomer_residues[0][[0, -1]])   # first and last, before the offset
```

Set the offset so that every model level lands on the same numbering as your
reference structure. `DOMAIN_SEGMENTS` is then given in those same real residue
numbers.

**Figures look cramped**
Lower `FONTSIZE` or raise the `FIGSIZE_*` values in the configuration cell.

---

## Notes on method

- **Smoothing** is applied for display only. RMSD and Rg traces are smoothed with
  a Savitzky-Golay filter so that overlaid traces stay readable; all statistics
  reported by `summary()` use the raw data.
- **Protomer averaging** assumes a homo-oligomer whose subunits appear as
  consecutive equal-length residue blocks. For a hetero-oligomer, set
  `N_PROTOMERS = 1` and slice the profiles manually instead.
- **Variability** is a plain standard deviation across a set of runs, so with
  only a few conditions per group it is a coarse estimate. Treat consensus peaks
  seen across most runs as more reliable than one-off hits.
- **Peak picking** flags candidate positions for follow-up, not conclusions.
  Corroborate with a structural inspection of the B-factor output.

---

## Repository housekeeping

This notebook is one of several in the repository, so a few conventions keep
them from interfering with each other.

**Shared, repository-level files.** `requirements.txt` and `.gitignore` live at
the repository root, one of each, covering every notebook. Neither can be
renamed per notebook: git only recognises the filename `.gitignore`, and
`pip install -r requirements.txt` is what every reader and CI runner expects.

**Per-notebook outputs.** Each notebook writes to `results/<NOTEBOOK_SLUG>/`,
set in the configuration cell, so two notebooks saving a figure of the same name
cannot overwrite one another.

**Shared input data.** `DATA_ROOT` and `demo_data/` are deliberately *not*
namespaced, so sibling notebooks read the same trajectories rather than each
keeping a copy.

**Sections 1-4 of the configuration cell are identical in every notebook.** If
you change the simulation grid, the protomer count or the domain boundaries,
change them the same way in each notebook. If that becomes tedious, move those
sections into a `channel_config.py` at the repository root and import it from
each notebook instead.

**Outputs are stripped** from the committed notebooks, which keeps diffs small
and stops absolute data paths leaking through cell output:

```bash
pip install nbstripout
nbstripout --install     # run once, inside the repository
```
